# DistilBERT Detector — Full Training Pipeline (Sequential Tokenization)

Complete training workflow for cross-generator generalization study with ablation studies.
No pre-tokenization caching—texts are tokenized sequentially during training.


In [1]:
from __future__ import annotations

import json
import os
import random
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, Subset
from transformers import AutoTokenizer, DistilBertModel

print("Libraries loaded", flush=True)

c:\Users\Rafay\Desktop\ANN PROJECT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded


In [2]:
ROOT_DIR      = Path.cwd()
if not (ROOT_DIR / "data").exists():
    ROOT_DIR = ROOT_DIR.parent

PROCESSED_DIR = ROOT_DIR / "data" / "processed"
ARTIFACT_DIR  = ROOT_DIR / "artifacts" / "distilbert_detector"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

SEED            = 42
EPOCHS          = 3
BATCH_SIZE      = 32
MAX_LENGTH      = 256
LR              = 2e-5
WEIGHT_DECAY    = 0.01
GRAD_CLIP_NORM  = 1.0
NUM_WORKERS     = 0  # Must be 0 in Jupyter
PIN_MEMORY      = torch.cuda.is_available()
TOKENIZER_NAME  = "distilbert-base-uncased"

SAMPLES_PER_CLASS = 10_000
UNSEEN_CAP        = 5_000

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Project root   : {ROOT_DIR}", flush=True)
print(f"Processed data : {PROCESSED_DIR}", flush=True)
print(f"Artifacts      : {ARTIFACT_DIR}", flush=True)
print(f"Device         : {DEVICE}", flush=True)
print(f"Workers        : {NUM_WORKERS}", flush=True)

Project root   : c:\Users\Rafay\Desktop\ANN PROJECT
Processed data : c:\Users\Rafay\Desktop\ANN PROJECT\data\processed
Artifacts      : c:\Users\Rafay\Desktop\ANN PROJECT\artifacts\distilbert_detector
Device         : cuda
Workers        : 0


In [3]:
def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print(f"Seed: {SEED}", flush=True)

Seed: 42


In [4]:
class DistilBertClassifier(nn.Module):
    def __init__(self, head_type: str = "single", freeze_layers: int = 0):
        super().__init__()
        if head_type not in {"single", "deep"}:
            raise ValueError("head_type must be 'single' or 'deep'")
        if not 0 <= freeze_layers <= 6:
            raise ValueError("freeze_layers must be between 0 and 6")

        self.head_type    = head_type
        self.freeze_layers = freeze_layers
        self.distilbert   = DistilBertModel.from_pretrained(TOKENIZER_NAME)
        hidden            = self.distilbert.config.hidden_size

        if head_type == "single":
            self.classifier = nn.Sequential(
                nn.Dropout(0.3),
                nn.Linear(hidden, 2),
            )
        else:
            self.classifier = nn.Sequential(
                nn.Dropout(0.3),
                nn.Linear(hidden, 384),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(384, 2),
            )

        self._freeze_layers()

    def _freeze_layers(self) -> None:
        for p in self.distilbert.embeddings.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.distilbert.transformer.layer):
            req = i >= self.freeze_layers
            for p in layer.parameters():
                p.requires_grad = req

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        out = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.classifier(cls)


def get_model_config(ablation: str) -> dict[str, Any]:
    configs = {
        "baseline1":  {"head_type": "single", "freeze_layers": 0},
        "ablation_a": {"head_type": "single", "freeze_layers": 4},
        "ablation_b": {"head_type": "deep",   "freeze_layers": 4},
        "ablation_c": {"head_type": "deep",   "freeze_layers": 0},
    }
    if ablation not in configs:
        raise ValueError(f"Unknown ablation: {ablation}")
    return configs[ablation]


def count_trainable(model: nn.Module) -> tuple[int, int]:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    return trainable, total


print("\nModel configurations:", flush=True)
for name in ["baseline1", "ablation_a", "ablation_b", "ablation_c"]:
    cfg = get_model_config(name)
    m   = DistilBertClassifier(**cfg)
    tr, tot = count_trainable(m)
    print(f"  {name:12s} -> {cfg} | trainable={tr:,} / total={tot:,}", flush=True)
    del m


Model configurations:


c:\Users\Rafay\Desktop\ANN PROJECT\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  baseline1    -> {'head_type': 'single', 'freeze_layers': 0} | trainable=42,528,770 / total=66,364,418
  ablation_a   -> {'head_type': 'single', 'freeze_layers': 4} | trainable=14,177,282 / total=66,364,418
  ablation_b   -> {'head_type': 'deep', 'freeze_layers': 4} | trainable=14,471,810 / total=66,658,946
  ablation_c   -> {'head_type': 'deep', 'freeze_layers': 0} | trainable=42,823,298 / total=66,658,946


In [5]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
print(f"Tokenizer loaded: {TOKENIZER_NAME}", flush=True)


class SequentialTokenDataset(Dataset):
    """Dataset that tokenizes texts on-the-fly during iteration."""
    
    def __init__(self, texts: list[str], labels: list[int], tokenizer, max_length: int = MAX_LENGTH):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self) -> int:
        return len(self.labels)
    
    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        text = self.texts[idx]
        label = self.labels[idx]
        
        enc = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(label, dtype=torch.long),
        }


print("Dataset class defined", flush=True)

Tokenizer loaded: distilbert-base-uncased
Dataset class defined


In [6]:
print("\nPreparing datasets...", flush=True)

train_parquet = PROCESSED_DIR / "train_pool.parquet"
capped_parquet = PROCESSED_DIR / "train_pool_capped.parquet"

if not capped_parquet.exists():
    print("Creating capped training dataset...", flush=True)
    df = pd.read_parquet(train_parquet)
    df_human = df[df["label"] == 0].sample(n=SAMPLES_PER_CLASS, random_state=SEED)
    df_ai    = df[df["label"] == 1].sample(n=SAMPLES_PER_CLASS, random_state=SEED)
    df_capped = (
        pd.concat([df_human, df_ai])
        .sample(frac=1, random_state=SEED)
        .reset_index(drop=True)
    )
    df_capped.to_parquet(capped_parquet, index=False)
    print(f"Saved capped dataset: {len(df_capped):,} rows", flush=True)
else:
    print(f"Capped dataset found", flush=True)

df_train = pd.read_parquet(capped_parquet)
print(f"Train dataset: {len(df_train):,} rows", flush=True)
print(df_train["label"].value_counts().to_string(), flush=True)

unseen_parquet = PROCESSED_DIR / "test_unseen.parquet"
df_unseen = pd.read_parquet(unseen_parquet)

n = min(UNSEEN_CAP // 2, 
        (df_unseen["label"] == 0).sum(),
        (df_unseen["label"] == 1).sum())
df_unseen_cap = pd.concat([
    df_unseen[df_unseen["label"] == 0].sample(n=n, random_state=SEED),
    df_unseen[df_unseen["label"] == 1].sample(n=n, random_state=SEED),
]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"\nUnseen dataset: {len(df_unseen_cap):,} rows (capped from {len(df_unseen):,})", flush=True)
print(df_unseen_cap["label"].value_counts().to_string(), flush=True)


Preparing datasets...
Creating capped training dataset...
Saved capped dataset: 120,000 rows
Train dataset: 120,000 rows
label
1    60000
0    60000

Unseen dataset: 10,000 rows (capped from 20,000)
label
1    5000
0    5000


In [7]:
print("\nBuilding DataLoaders...", flush=True)

texts_train = df_train["text"].tolist()
labels_train = df_train["label"].tolist()

indices = list(range(len(texts_train)))
train_idx, temp_idx = train_test_split(
    indices, test_size=0.2, random_state=SEED, stratify=labels_train
)
temp_labels = [labels_train[i] for i in temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, random_state=SEED, stratify=temp_labels
)

texts_train_set    = [texts_train[i] for i in train_idx]
labels_train_set   = [labels_train[i] for i in train_idx]
texts_val_set      = [texts_train[i] for i in val_idx]
labels_val_set     = [labels_train[i] for i in val_idx]
texts_test_set     = [texts_train[i] for i in test_idx]
labels_test_set    = [labels_train[i] for i in test_idx]
texts_unseen_set   = df_unseen_cap["text"].tolist()
labels_unseen_set  = df_unseen_cap["label"].tolist()

train_ds  = SequentialTokenDataset(texts_train_set, labels_train_set, tokenizer, MAX_LENGTH)
val_ds    = SequentialTokenDataset(texts_val_set, labels_val_set, tokenizer, MAX_LENGTH)
test_ds   = SequentialTokenDataset(texts_test_set, labels_test_set, tokenizer, MAX_LENGTH)
unseen_ds = SequentialTokenDataset(texts_unseen_set, labels_unseen_set, tokenizer, MAX_LENGTH)

kw = {"num_workers": NUM_WORKERS, "pin_memory": PIN_MEMORY}

train_loader  = DataLoader(train_ds,  batch_size=BATCH_SIZE, shuffle=True,  **kw)
val_loader    = DataLoader(val_ds,    batch_size=BATCH_SIZE, shuffle=False, **kw)
test_loader   = DataLoader(test_ds,   batch_size=BATCH_SIZE, shuffle=False, **kw)
unseen_loader = DataLoader(unseen_ds, batch_size=BATCH_SIZE, shuffle=False, **kw)

print(f"Train:  {len(train_ds):,} samples → {len(train_loader)} batches", flush=True)
print(f"Val:    {len(val_ds):,} samples → {len(val_loader)} batches", flush=True)
print(f"Test:   {len(test_ds):,} samples → {len(test_loader)} batches", flush=True)
print(f"Unseen: {len(unseen_ds):,} samples → {len(unseen_loader)} batches", flush=True)


Building DataLoaders...
Train:  96,000 samples → 3000 batches
Val:    12,000 samples → 375 batches
Test:   12,000 samples → 375 batches
Unseen: 10,000 samples → 313 batches


In [8]:
def compute_metrics(
    labels: list[int],
    preds: list[int],
    probs: list[float],
    loss: float,
) -> dict[str, float]:
    metrics = {
        "accuracy":  accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall":    recall_score(labels, preds, zero_division=0),
        "f1":        f1_score(labels, preds, zero_division=0),
        "loss":      loss,
    }
    try:
        metrics["roc_auc"] = roc_auc_score(labels, probs)
    except ValueError:
        metrics["roc_auc"] = float("nan")
    return metrics


def move_batch(batch: dict, device: torch.device) -> dict:
    return {k: v.to(device) for k, v in batch.items()}


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer | None = None,
) -> dict[str, float]:
    training = optimizer is not None
    model.train() if training else model.eval()

    criterion   = nn.CrossEntropyLoss()
    running_loss = 0.0
    all_labels: list[int]  = []
    all_preds:  list[int]  = []
    all_probs:  list[float] = []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch in loader:
            batch = move_batch(batch, DEVICE)
            logits = model(batch["input_ids"], batch["attention_mask"])
            loss   = criterion(logits, batch["labels"])

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                optimizer.step()

            running_loss += loss.item() * batch["labels"].size(0)
            probs  = torch.softmax(logits, dim=1)[:, 1].detach().cpu().tolist()
            preds  = logits.argmax(dim=1).detach().cpu().tolist()
            labels = batch["labels"].cpu().tolist()

            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels)

    avg_loss = running_loss / len(loader.dataset)
    return compute_metrics(all_labels, all_preds, all_probs, avg_loss)


print("Training functions defined", flush=True)

Training functions defined


In [ ]:
def train_model(
    ablation_name: str,
    train_loader: DataLoader,
    val_loader:   DataLoader,
    test_loader:  DataLoader,
    unseen_loader: DataLoader,
) -> dict[str, Any]:
    config = get_model_config(ablation_name)
    model  = DistilBertClassifier(**config).to(DEVICE)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    best_val_f1 = -1.0
    best_state: dict | None = None
    history: list[dict] = []

    for epoch in range(1, EPOCHS + 1):
        print(f"\n{'='*60}", flush=True)
        print(f"[{ablation_name}] EPOCH {epoch}/{EPOCHS} START", flush=True)
        epoch_start = time.perf_counter()
        
        t0 = time.perf_counter()
        train_metrics = run_epoch(model, train_loader, optimizer)
        train_time    = round(time.perf_counter() - t0, 2)
        print(f"  → Train complete: {train_time}s", flush=True)

        t1 = time.perf_counter()
        val_metrics = run_epoch(model, val_loader)
        val_time    = round(time.perf_counter() - t1, 2)
        print(f"  → Val complete: {val_time}s", flush=True)
        
        total_epoch_time = round(time.perf_counter() - epoch_start, 2)
        print(f"[{ablation_name}] EPOCH {epoch}/{EPOCHS} END — Total: {total_epoch_time}s", flush=True)
        print(f"{'='*60}", flush=True)

        history.append({
            "epoch": epoch,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}":   v for k, v in val_metrics.items()},
        })

        print(
            f"[{ablation_name}] epoch={epoch}/{EPOCHS} "
            f"train={train_metrics} val={val_metrics}",
            flush=True,
        )

        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_state  = {
                "model_state_dict": model.state_dict(),
                "config":           config,
                "ablation_name":    ablation_name,
                "epoch":            epoch,
                "val_metrics":      val_metrics,
            }

    assert best_state is not None
    model.load_state_dict(best_state["model_state_dict"])

    print(f"\n[{ablation_name}] Running test evaluation…", flush=True)
    test_metrics = run_epoch(model, test_loader)

    print(f"[{ablation_name}] Running unseen evaluation…", flush=True)
    unseen_metrics = run_epoch(model, unseen_loader)

    return {
        "ablation_name":   ablation_name,
        "config":          config,
        "history":         history,
        "best_val_f1":     best_val_f1,
        "test_metrics":    test_metrics,
        "unseen_metrics":  unseen_metrics,

        "model_state_dict": model.state_dict(),    }

In [ ]:
print("\n" + "="*60, flush=True)
print("RUNNING ALL ABLATIONS", flush=True)
print("="*60, flush=True)

results: dict[str, Any] = {}
training_times: dict[str, float] = {}

ablation_name = "ablation_b"
print(f"\n{'='*60}", flush=True)
print(f"Starting ablation: {ablation_name}", flush=True)
print(f"Config           : {get_model_config(ablation_name)}", flush=True)
print(f"{'='*60}", flush=True)

t_start = time.perf_counter()
results[ablation_name] = train_model(
    ablation_name,
    train_loader,
    val_loader,
    test_loader,
    unseen_loader,
)
elapsed = round(time.perf_counter() - t_start, 2)
training_times[ablation_name] = elapsed
print(
    f"\n[{ablation_name}] total training time: {elapsed}s "
    f"({elapsed/3600:.2f} hrs)",
    flush=True,
)


RUNNING ALL ABLATIONS

Starting ablation: baseline1
Config           : {'head_type': 'single', 'freeze_layers': 0}


c:\Users\Rafay\Desktop\ANN PROJECT\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



[baseline1] epoch=1/3 starting train…


KeyboardInterrupt: 

In [ ]:
summary_rows = []
for name, result in results.items():
    row = {
        "ablation_name": name,
        "best_val_f1": result["best_val_f1"],
        **{f"test_{k}": v   for k, v in result["test_metrics"].items()},
        **{f"unseen_{k}": v for k, v in result["unseen_metrics"].items()},
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print("\n" + "=" * 60, flush=True)
print("FINAL SUMMARY", flush=True)
print("=" * 60, flush=True)
print(summary_df.to_string(index=False), flush=True)

print("\nRelative Robustness Degradation (RRD):", flush=True)
for _, row in summary_df.iterrows():
    seen_f1   = row["test_f1"]
    unseen_f1 = row["unseen_f1"]
    rrd = (seen_f1 - unseen_f1) / seen_f1 * 100 if seen_f1 > 0 else float("nan")
    print(f"  {row['ablation_name']:12s}: seen_F1={seen_f1:.4f}  unseen_F1={unseen_f1:.4f}  RRD={rrd:.2f}%", flush=True)

best_name = summary_df.sort_values("best_val_f1", ascending=False).iloc[0]["ablation_name"]

print("\n" + "=" * 60, flush=True)
print("SAVING ARTIFACTS", flush=True)
print("=" * 60, flush=True)

for name, result in results.items():
    ckpt_path = ARTIFACT_DIR / f"{name}_best.pt"
    torch.save(result, ckpt_path)
    print(f"Saved: {ckpt_path}", flush=True)

with open(ARTIFACT_DIR / "training_times.json", "w", encoding="utf-8") as f:
    json.dump(training_times, f, indent=2)

with open(ARTIFACT_DIR / "best_config.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "best_ablation_name": best_name,
            "config": results[best_name]["config"],
            "best_val_f1": float(results[best_name]["best_val_f1"]),
        },
        f,
        indent=2,
    )

tokenizer.save_pretrained(ARTIFACT_DIR)
summary_df.to_csv(ARTIFACT_DIR / "summary.csv", index=False)

print(f"\nAll artifacts saved to {ARTIFACT_DIR}", flush=True)
print(f"Best ablation: {best_name}", flush=True)
print(f"\nDone!", flush=True)